In [1]:
pip install pandas_ta

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 50.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.43.0
    Uninstalling llvmlite-0.43.0:
      Successfully uninstalled llvmlite-0.43.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninsta

In [1]:
import math
import random
from datetime import datetime, timedelta
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas_ta as ta
import copy

In [2]:
pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.1 MB/s eta 0:00:00


In [3]:
# Install and import PyG modules separately; user must have torch-geometric installed.
try:
    from torch_geometric.nn import GCNConv
except Exception as e:
    raise ImportError("Please install torch-geometric following the official instructions. Error: " + str(e))

# Transformers FinBERT
from transformers import pipeline

# yfinance for price data
import yfinance as yf

### HSBC

In [4]:
from google.colab import drive

drive.mount('/content/gdrive')

apple_data = pd.read_excel(r"/content/gdrive/My Drive/Dataset/News_Info_part2.xlsx", sheet_name="HSBC")
apple_data['date']=pd.to_datetime(apple_data['date']).dt.date

apple_data.dropna(inplace=True)
apple_data

Mounted at /content/gdrive


,date,title,content,link,symbols,tags,sentiment
0,2024-05-08,HSBC Holdings PLC's Dividend Analysis,Exploring the Sustainability of HSBC Holdings ...,https://finance.yahoo.com/news/hsbc-holdings-p...,"['0005.HK', 'H1SB34.SA', 'HBC1.BE', 'HBC1.DU',...",[],"{'polarity': 0.998, 'neg': 0.014, 'neu': 0.861..."
1,2024-05-06,Top Three UK Dividend Stocks To Watch In May 2024,As the FTSE 100 mirrors a positive trend in gl...,https://finance.yahoo.com/news/top-three-uk-di...,"['0005.HK', 'BYG.LSE', 'DCC.LSE', 'GFTU.LSE', ...",[],"{'polarity': 0.998, 'neg': 0.013, 'neu': 0.879..."
2,2024-05-04,Ping An votes against reappointment of HSBC CE...,(Reuters) - HSBC Holdings Plc's biggest Asian ...,https://finance.yahoo.com/news/ping-votes-agai...,"['0005.HK', '2318.HK', '601318.SHG', 'H1SB34.S...",[],"{'polarity': -0.477, 'neg': 0.044, 'neu': 0.93..."
3,2024-05-04,HSBC Holdings First Quarter 2024 Earnings: Bea...,HSBC Holdings (LON:HSBA) First Quarter 2024 Re...,https://finance.yahoo.com/news/hsbc-holdings-f...,"['0005.HK', 'H1SB34.SA', 'HBC1.BE', 'HBC1.DU',...",[],"{'polarity': 0.776, 'neg': 0.035, 'neu': 0.911..."
4,2024-05-03,3 Foreign Bank Stocks Worth a Look in a Prospe...,Banks across the globe have been continuously ...,https://finance.yahoo.com/news/3-foreign-bank-...,"['0005.HK', 'GSPC.INDX', 'H1SB34.SA', 'HBC1.BE...",[],"{'polarity': 1, 'neg': 0.013, 'neu': 0.852, 'p..."
...,...,...,...,...,...,...,...
1909,2017-10-17,HSBC Holdings plc Sponsored ADR (HSBC) a Buy o...,"Currently, HSBC Holdings plc Sponsored ADR (NY...",https://investorplace.com/2017/10/hsbc-holding...,['HSBC.US'],[],"{'polarity': 0.296, 'neg': 0, 'neu': 0.945, 'p..."
1910,2017-10-10,HSBC Holdings plc Sponsored ADR (HSBC) Rating ...,HSBC Holdings plc Sponsored ADR (NYSE:HSBC) is...,https://investorplace.com/2017/10/hsbc-holding...,['HSBC.US'],[],"{'polarity': 0.671, 'neg': 0, 'neu': 0.857, 'p..."
1911,2017-09-25,5 Mega-Cap Stocks to Buy with Impressive Growt...,With the U.S. Fed deciding to turn off the qua...,https://investorplace.com/2017/09/5-mega-cap-s...,"['BAYRY.US', 'HSBC.US']",[],"{'polarity': 0.863, 'neg': 0, 'neu': 0.71, 'po..."
1912,2017-08-07,10 Great Vanguard Funds for Your Income Portfolio,These 10 great Vanguard funds offer low fees a...,https://investorplace.com/2017/08/10-great-van...,['HSBC.US'],[],"{'polarity': 0.827, 'neg': 0.066, 'neu': 0.627..."


In [5]:
unique_tickers = pd.read_excel(r"/content/gdrive/My Drive/Dataset/fixed nodes.xlsx")['HSBC'].dropna()
unique_tickers = list(unique_tickers)
unique_tickers

['HSBC',
 'EPS',
 'AGM',
 'PLUS',
 'RIO',
 'BCS',
 'CAGR',
 'DB',
 'EU',
 'EPC',
 'BHP',
 'BST',
 'GSK',
 'UAE',
 'ECL',
 'GS',
 'KKR',
 'AAPL',
 'AMKBY',
 'AMZN',
 'EBAY',
 'EXPE',
 'KHC',
 'KO',
 'LVMUY',
 'MAR',
 'MCD',
 'NVO',
 'PYPL',
 'SBUX',
 'STLA',
 'UBS',
 'CMG',
 'ORCL',
 'RTX',
 'HRB',
 'INTU',
 'IP',
 'IPO',
 'FCA',
 'MPC',
 'TSM',
 'RBC',
 'TD',
 'C',
 'RY',
 'MUFG',
 'BOE',
 'CFO',
 'AER',
 'FAB',
 'BAC',
 'BN',
 'MS',
 'SPGI',
 'TM',
 'JPM',
 'ADI',
 'EXC',
 'NVDA',
 'ETSY',
 'SAN',
 'ETR',
 'GOOG',
 'IBM',
 'MET',
 'MSFT',
 'PRU',
 'OCC',
 'WFC',
 'SHEL',
 'ESG',
 'AZN',
 'ALLY',
 'SYF',
 'FSCS',
 'WASH',
 'AIA',
 'JNJ',
 'META',
 'AP',
 'AMD',
 'TSLA',
 'CAC',
 'BTI',
 'VOD',
 'BABA',
 'BIDU',
 'FXI',
 'JD',
 'KWEB',
 'TCEHY',
 'SNEX',
 'USD',
 'BMO',
 'VC',
 'IBN',
 'BNPQY',
 'DELL',
 'GM',
 'NKE',
 'FSS',
 'EQNR',
 'LPG',
 'SNY',
 'TTE',
 'UL',
 'BYD',
 'BYDDF',
 'BLK',
 'STT',
 'RYAAY',
 'TMUS',
 'NWG',
 'RITM',
 'REI',
 'UNCFF',
 'BP',
 'CB',
 'ETN',
 'LIN',
 'MDT

In [6]:
import math
import random
from datetime import datetime, timedelta
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
# -------------------------
# Utilities
# -------------------------
def returns_from_prices(prices: np.ndarray) -> np.ndarray:
    """Compute log returns. prices shape (T, N) -> returns (T-1, N)"""
    return np.log(prices[1:] / prices[:-1] + 1e-12)

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import pipeline


from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

def finbert_sentiment_pipeline(device=-1):
    model_path = "/content/gdrive/My Drive/finbert"

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)

    return pipeline(
        "sentiment-analysis",
        model=model,
        tokenizer=tokenizer,
        device=device
    )

def classify_news_sentiment(news_df: pd.DataFrame, pipe) -> pd.DataFrame:
    """
    Run FinBERT sentiment on each news text and return the news_df with 'label' and 'score' columns.
    label usually one of: 'positive', 'neutral', 'negative' (model dependent); we will normalize later.
    """
    texts = news_df["title"].tolist()
    # pipe can process batches; to be safe, process in small batches
    batch_size = 16
    labels = []
    scores = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        outs = pipe(batch)
        # outs: list of {'label': 'positive', 'score': 0.98} etc.
        for o in outs:
            labels.append(o["label"])
            scores.append(float(o.get("score", 0.0)))
    df = news_df.copy().reset_index(drop=True)
    df["label"] = labels
    df["score"] = scores
    return df

import ast
def parse_mentions(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return x

def aggregate_sentiment_per_company_day(news_df_with_labels: pd.DataFrame, tickers: List[str], date_index: pd.DatetimeIndex):
    """
    For each (date, ticker) compute an aggregated sentiment score.
    Strategy:
      - Map labels to numeric: positive -> +1, neutral -> 0, negative -> -1
      - Use label * score as weighted value, average across news mentioning the ticker (normalized format) on that date.
    Returns:
      sentiment_df: DataFrame indexed by date_index (dates from price_df) with columns tickers (shape T x N)
    """

    def label_to_num(lab):
        if pd.isna(lab):
            return 0.0
        s = str(lab).lower()
        if "pos" in s:
            return 1.0
        elif "neg" in s:
            return -1.0
        else:
            return 0.0

    tmp = news_df_with_labels.copy()
    #tmp["date"] = pd.to_datetime(tmp["date"]).dt.normalize()

    # Ensure mentions are parsed lists
    if isinstance(tmp["mentions"].iloc[0], str):
        tmp["mentions"] = tmp["mentions"].apply(lambda x: eval(x) if isinstance(x, str) else x)

    # Normalize tickers in 'mentions' — e.g. "AAPL.US" -> "AAPL"
    def normalize_ticker(name):
        if not isinstance(name, str):
            return name
        return name.split(".")[0].upper().strip()
    


    tmp["mentions"] = tmp["mentions"].apply(lambda lst: [normalize_ticker(x) for x in lst])

    # Aggregate sentiment values
    accum = {}
    for _, row in tmp.iterrows():
        d = row["date"]
        numeric = label_to_num(row["label"]) * float(row["score"])
        for t in row["mentions"]:
            if t in tickers:  # only keep known companies
                accum.setdefault((d, t), []).append(numeric)

    # Build sentiment matrix
    rows = []
    for d in date_index:
        row_vals = []
        for t in tickers:
            vals = accum.get((d, t), [])
            row_vals.append(float(np.mean(vals)) if vals else 0.0)
        rows.append(row_vals)

    sentiment_df = pd.DataFrame(rows, index=date_index, columns=tickers).astype(np.float32)
    return sentiment_df


# -------------------------
# Graph construction from news co-occurrence
# -------------------------

def normalize_ticker(name):
    if not isinstance(name, str):
        return name
    return name.split(".")[0].upper().strip()




In [8]:
def return_correlation_graph_for_day(
    returns_df: pd.DataFrame,
    tickers: List[str],
    target_idx: int,
    window: int = 5,
    min_edge_weight: float = 0.1,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build an edge index and weight matrix based on the absolute Pearson correlation
    of stock returns over a rolling window ending at target_idx.
    """
    n = len(tickers)

    # 1. Slice the rolling window up to the target index
    start_idx = max(0, target_idx - window + 1)
    window_returns = returns_df.iloc[start_idx: target_idx + 1][tickers]

    # 2. Compute the Pearson correlation matrix
    if len(window_returns) > 2:
        # fillna(0.0) handles cases with zero variance
        corr_mat = window_returns.corr().fillna(0.0).values
    else:
        # Fallback if there aren't enough lookback steps yet
        corr_mat = np.zeros((n, n), dtype=np.float32)

    # 3. Use absolute correlation to define edge strength
    mat = np.abs(corr_mat)

    # Remove self-loops
    np.fill_diagonal(mat, 0.0)

    # 4. Filter out weak connections
    mask = mat >= min_edge_weight

    if not np.any(mask):
        return np.zeros((2, 0), dtype=np.int64), np.zeros((0,), dtype=np.float32)

    rows, cols = np.nonzero(mask)

    edges = np.stack([rows, cols], axis=0).astype(np.int64)
    weights = mat[rows, cols].astype(np.float32)

    return edges, weights

In [9]:
class DynamicGraphDatasetNews(torch.utils.data.Dataset):
    def __init__(
        self,
        price_df: pd.DataFrame,
        news_df: pd.DataFrame,
        sentiment_df: pd.DataFrame,
        seq_len: int = 5,
        co_window: int = 5,
        mode: str = "regression",
    ):
        assert mode in ("regression", "classification")

        self.seq_len = seq_len
        self.co_window = co_window
        self.mode = mode

        prices = price_df.values.astype(np.float32)
        dates = pd.to_datetime(price_df.index)
        self.date_index = dates
        self.tickers = list(price_df.columns)

        T, N = prices.shape

        sentiment_df = sentiment_df.reindex(dates).fillna(0.0)
        svals = sentiment_df.values.astype(np.float32)

        # --------------------------------------------------
        # PRE-COMPUTE DAILY LOG RETURNS
        # --------------------------------------------------
        self.returns_df = np.log(
            (price_df + 1e-8) / (price_df.shift(1) + 1e-8)
        ).fillna(0.0)

        returns = self.returns_df.values.astype(np.float32)

           # ---- PRICE NORMALIZATION  ----
        W = 252
        VOL_WINDOW = 20
        T, N = prices.shape


        # Initialize arrays to the full length T
        norm_prices = np.zeros((T, N), dtype=np.float32)
        vol_norm = np.zeros((T, N), dtype=np.float32)
        targets = np.zeros((T, N), dtype=np.float32)

        # ---- PROCESSING ALL TIMESTEPS ----
        for t in range(T):
            # 1. Determine the window bounds
            # If t < W, use everything from 0 to t (Expanding)
            # If t >= W, use t-W+1 to t (Rolling)
            start_idx = max(0, t - W + 1)
            window = prices[start_idx : t + 1]

            # 2. Calculate statistics
            mean_t = window.mean(axis=0)
            std_t = window.std(axis=0) + 1e-6

            # 3. Normalize Current Price
            norm_prices[t] = (prices[t] - mean_t) / std_t

            # 4. Normalize Target Price (Price at t+1)
            # We can only do this if t < T-1
            if t < T - 1:
                targets[t] = (prices[t+1] - mean_t) / std_t


          # ----- Volatility -----
            vol_start = max(0, t - VOL_WINDOW + 1)
            return_window = returns[vol_start:t+1]

            # Annualize
            current_vol = return_window.std(axis=0) * np.sqrt(252) # Shape (N,)

            vol_norm[t] = current_vol

        # --------------------------------------------------
        # FEATURE CONSTRUCTION
        # --------------------------------------------------
        feature_list = []

        for t in range(T - 1):

            feat_t = np.stack(
                [
                    norm_prices[t],   # Normalized price
                    svals[t],         # Sentiment
                    vol_norm[t],      # Rolling volatility
                ],
                axis=1,
            )

            feature_list.append(feat_t.astype(np.float32))

        self.features = np.stack(feature_list, axis=0)
        self.targets = targets[:T - 1]

        self.valid_end_idx = list(range(self.seq_len - 1, T - 1))

        # --------------------------------------------------
        # GRAPH CONSTRUCTION (RETURN CORRELATION)
        # --------------------------------------------------
        self.edge_index_list = []
        self.edge_weight_list = []

        for t in range(T - 1):

            ei, ew = return_correlation_graph_for_day(
                returns_df=self.returns_df,
                tickers=self.tickers,
                target_idx=t,
                window=self.co_window,
                min_edge_weight=0.1,
            )

            # ------------------------------------------
            # EDGE WEIGHT NORMALIZATION
            # ------------------------------------------
            if ew is not None and len(ew) > 0:

                ew = ew.astype(np.float32)


            self.edge_index_list.append(ei)
            self.edge_weight_list.append(ew)

    def __len__(self):
        return len(self.valid_end_idx)

    def __getitem__(self, idx):

        end_t = self.valid_end_idx[idx]
        start_t = end_t - (self.seq_len - 1)

        seq_feats = self.features[start_t:end_t + 1]
        seq_edge_idx = self.edge_index_list[start_t:end_t + 1]
        seq_edge_w = self.edge_weight_list[start_t:end_t + 1]

        target = self.targets[end_t]

        if self.mode == "classification":
            y = (target > 0).astype(np.int64)
        else:
            y = target.astype(np.float32)

        return {
            "seq_feats": torch.from_numpy(seq_feats),
            "seq_edge_index": seq_edge_idx,
            "seq_edge_weight": seq_edge_w,
            "target": torch.from_numpy(y),
        }

In [10]:


def collate_dynamic(batch):
    seq_feats = torch.stack([item["seq_feats"] for item in batch], dim=0)
    targets = torch.stack([item["target"] for item in batch], dim=0)
    seq_edge_index = [item["seq_edge_index"] for item in batch]
    seq_edge_weight = [item["seq_edge_weight"] for item in batch]

    return {
        "seq_feats": seq_feats,
        "seq_edge_index": seq_edge_index,
        "seq_edge_weight": seq_edge_weight,
        "target": targets,
    }

class TGCN(nn.Module):
    def __init__(self, in_feats, gcn_hidden=64, gru_hidden=64, out_dim=1, dropout=0.2, mode="regression"):
        super().__init__()
        self.mode = mode

        self.gcn1 = GCNConv(in_feats, gcn_hidden)
        self.gcn2 = GCNConv(gcn_hidden, gcn_hidden)

        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(gcn_hidden, gru_hidden, batch_first=False)

        self.mlp = nn.Sequential(
            nn.Linear(gru_hidden, gru_hidden // 2),
            nn.ReLU(),
            nn.Linear(gru_hidden // 2, out_dim),
        )

    def forward(self, seq_feats, seq_edge_index, seq_edge_weight):
        batch, seq_len, N, F_dim = seq_feats.shape
        device = seq_feats.device

        gcn_outputs = []

        for t in range(seq_len):
            batch_node_embeds = []

            for b in range(batch):
                x = seq_feats[b, t].to(device)

                ei_np = seq_edge_index[b][t]
                ew_np = seq_edge_weight[b][t]

                if ei_np is None or len(ei_np) == 0:
                    edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
                    edge_weight = None
                else:
                    edge_index = torch.from_numpy(ei_np).long().to(device)

                    if ew_np is not None:
                        edge_weight = torch.from_numpy(ew_np).float().to(device)
                    else:
                        edge_weight = None

                h = torch.relu(self.gcn1(x, edge_index, edge_weight))
                h = self.dropout(h)
                h = torch.relu(self.gcn2(h, edge_index, edge_weight))

                batch_node_embeds.append(h)

            gcn_outputs.append(torch.stack(batch_node_embeds, dim=0))

        seq_stack = torch.stack(gcn_outputs, dim=0)  # (seq_len, batch, N, hidden)
        seq_flat = seq_stack.view(seq_len, batch * N, -1)

        gru_out, _ = self.gru(seq_flat)
        last = gru_out[-1]

        preds = self.mlp(last)
        preds = preds.view(batch, N, -1)

        if self.mode == "classification":
            preds = torch.sigmoid(preds)

        return preds.squeeze(-1)

def train_epoch(
    model,
    loader,
    optimizer,
    device,
    loss_fn,
    company_idx=0,
):
    model.train()
    total_loss = 0.0

    for batch in loader:
        seq_feats = batch["seq_feats"].to(device)
        seq_edge_index = batch["seq_edge_index"]
        seq_edge_weight = batch["seq_edge_weight"]
        target = batch["target"].to(device)

        optimizer.zero_grad()

        preds = model(seq_feats, seq_edge_index, seq_edge_weight)

        # Compute loss only for the target company

        loss = loss_fn(
                preds[:, company_idx],
                target[:, company_idx]
            )


        loss.backward()
        optimizer.step()

        total_loss += loss.item() * seq_feats.size(0)

    return total_loss / len(loader.dataset)


def eval_epoch(
    model,
    loader,
    device,
    loss_fn,
    mode="regression",
    company_idx=0,
):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch in loader:
            seq_feats = batch["seq_feats"].to(device)
            seq_edge_index = batch["seq_edge_index"]
            seq_edge_weight = batch["seq_edge_weight"]
            target = batch["target"].to(device)

            preds = model(seq_feats, seq_edge_index, seq_edge_weight)

            # Keep the original loss for monitoring training
            loss = loss_fn(
                preds[:, company_idx],
                target[:, company_idx]
            )
            total_loss += loss.item() * seq_feats.size(0)

            all_preds.append(preds.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    preds = np.concatenate(all_preds, axis=0)
    targ = np.concatenate(all_targets, axis=0)

    # Evaluate only the target company
    if company_idx is not None:
        preds = preds[:, company_idx]
        targ = targ[:, company_idx]

    if mode == "classification":
        bin_preds = (preds > 0.5).astype(int)
        acc = (bin_preds == targ).mean()
        return avg_loss, {"accuracy": acc}, preds, targ
    else:
        mse = np.mean((preds - targ) ** 2)
        mae = np.mean(np.abs(preds - targ))
        return avg_loss, {"mse": mse, "mae": mae}, preds, targ

In [ ]:



def collate_dynamic(batch):
    seq_feats = torch.stack([item["seq_feats"] for item in batch], dim=0)
    targets = torch.stack([item["target"] for item in batch], dim=0)
    seq_edge_index = [item["seq_edge_index"] for item in batch]
    seq_edge_weight = [item["seq_edge_weight"] for item in batch]

    return {
        "seq_feats": seq_feats,
        "seq_edge_index": seq_edge_index,
        "seq_edge_weight": seq_edge_weight,
        "target": targets,
    }

class TGCN(nn.Module):
    def __init__(self, in_feats, gcn_hidden=64, gru_hidden=64, out_dim=1, dropout=0.0, mode="regression"):
        super().__init__()
        self.mode = mode

        self.gcn1 = GCNConv(in_feats, gcn_hidden)
        self.gcn2 = GCNConv(gcn_hidden, gcn_hidden)

        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(gcn_hidden, gru_hidden, batch_first=False)

        self.mlp = nn.Sequential(
            nn.Linear(gru_hidden, gru_hidden // 2),
            nn.ReLU(),
            nn.Linear(gru_hidden // 2, out_dim),
        )

    def forward(self, seq_feats, seq_edge_index, seq_edge_weight):
        batch, seq_len, N, F_dim = seq_feats.shape
        device = seq_feats.device

        gcn_outputs = []

        for t in range(seq_len):
            batch_node_embeds = []

            for b in range(batch):
                x = seq_feats[b, t].to(device)

                ei_np = seq_edge_index[b][t]
                ew_np = seq_edge_weight[b][t]

                if ei_np is None or len(ei_np) == 0:
                    edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
                    edge_weight = None
                else:
                    edge_index = torch.from_numpy(ei_np).long().to(device)

                    if ew_np is not None:
                        edge_weight = torch.from_numpy(ew_np).float().to(device)
                    else:
                        edge_weight = None

                h = torch.relu(self.gcn1(x, edge_index, edge_weight))
                h = self.dropout(h)
                h = torch.relu(self.gcn2(h, edge_index, edge_weight))

                batch_node_embeds.append(h)

            gcn_outputs.append(torch.stack(batch_node_embeds, dim=0))

        seq_stack = torch.stack(gcn_outputs, dim=0)  # (seq_len, batch, N, hidden)
        seq_flat = seq_stack.view(seq_len, batch * N, -1)

        gru_out, _ = self.gru(seq_flat)
        last = gru_out[-1]

        preds = self.mlp(last)
        preds = preds.view(batch, N, -1)

        if self.mode == "classification":
            preds = torch.sigmoid(preds)

        return preds.squeeze(-1)

def full_news_demo(
    tickers: List[str] = None,
    start_date: str = apple_data["date"].min(),
    end_date: str = apple_data["date"].max(),
    n_news: int = len(apple_data),
    seq_len: int = 5,
    co_window: int = 5,
    batch_size: int = 16,
    epochs: int = 6,
    company_idx: int = 0,
    device_str: str = "cpu",
):
    device = torch.device(device_str)
    if tickers is None:
        # default small set (pick companies with yfinance tickers)
        tickers = ["AAPL", "AMZN", "GOOGL", "TSLA", "NFLX", "MSFT", "META", "005930.KS", "CMCSA", "IT"]

    print("Downloading price data with yfinance...")
    price_df = yf.download(tickers, start=start_date, end=end_date, progress=False)["Close"]

    price_df = price_df[tickers]
    price_df = price_df.dropna(how="all").ffill().dropna(axis=1)  # drop tickers with no data

    news_df = apple_data[['date', 'title', 'symbols']]  # Include symbols for mentions
    news_df.rename(columns={'symbols': 'mentions'}, inplace=True)
    news_df["mentions"] = news_df["mentions"].apply(parse_mentions)

    pipe = finbert_sentiment_pipeline()  # cpu; set device=0 for GPU
    print("Classifying synthetic news with FinBERT...")
    news_labeled = classify_news_sentiment(news_df, pipe)
    news_labeled.reset_index(inplace=True, drop=True)  # Reset index to make date a column again
    print("Sample labeled news:")
    print(news_labeled.head())

    # Aggregate sentiment per date-company
    print("Aggregating sentiment per company-day...")
    date_index = price_df.index.date
    sentiment_df = aggregate_sentiment_per_company_day(news_labeled, list(price_df.columns), date_index)
    print("Sentiment df head:")
    print(sentiment_df.head())

    # build dataset
    print("Building DynamicGraphDatasetNews...")
    dataset = DynamicGraphDatasetNews(
        price_df=price_df,
        news_df=news_labeled,
        sentiment_df=sentiment_df,
        seq_len=seq_len,
        co_window=co_window,
        mode="regression"
    )
    n = len(dataset)
    train_n = int(0.7 * n)
    val_n = int(0.15 * n)
    idxs = list(range(n))
    train_idx = idxs[:train_n]
    val_idx = idxs[train_n : train_n + val_n]
    test_idx = idxs[train_n + val_n :]

    from torch.utils.data import Subset, DataLoader

    train_loader = DataLoader(Subset(dataset, train_idx), batch_size=batch_size, shuffle=False, collate_fn=collate_dynamic)
    val_loader = DataLoader(Subset(dataset, val_idx), batch_size=batch_size, shuffle=False, collate_fn=collate_dynamic)
    test_loader = DataLoader(Subset(dataset, test_idx), batch_size=batch_size, shuffle=False, collate_fn=collate_dynamic)

    in_feats = dataset.features.shape[-1]
    print(f"in_feats={in_feats}, dataset length={len(dataset)}")

    model = TGCN(
        in_feats=in_feats,
        gcn_hidden=64,
        gru_hidden=64,
        out_dim=1,
        dropout=0.2,
        mode="regression",
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5,
    )

    loss_fn = nn.MSELoss()

    # ------------------------------------------------------------------
    # Early stopping
    # ------------------------------------------------------------------

    patience = 5
    min_delta = 1e-4

    best_val_mse = float("inf")
    best_model = None
    best_epoch = 0
    counter = 0
    terminated_epoch = epochs  # Default to max epochs if no early stop occurs

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------

    for epoch in range(1, epochs + 1):

        train_loss = train_epoch(
            model,
            train_loader,
            optimizer,
            device,
            loss_fn,
            company_idx=company_idx,
        )

        val_loss, val_metrics, _, _ = eval_epoch(
            model,
            val_loader,
            device,
            loss_fn,
            mode="regression",
            company_idx=company_idx,
        )

        current_val_mse = val_metrics["mse"]

        print(
            f"Epoch {epoch}/{epochs} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f} | "
            f"Val MSE (Target Company): {current_val_mse:.6f}"
        )

        if current_val_mse < best_val_mse - min_delta:
            best_val_mse = current_val_mse
            best_model = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            counter = 0

            print(
                f"Validation improved. "
                f"Best validation MSE = {best_val_mse:.6f}"
            )

        else:
            counter += 1
            print(f"No improvement ({counter}/{patience})")

            if counter >= patience:
                terminated_epoch = epoch
                print(f"Early stopping triggered for company_idx={company_idx} at epoch {terminated_epoch}")
                break

    # ------------------------------------------------------------------
    # Restore best model
    # ------------------------------------------------------------------

    if best_model is not None:
        model.load_state_dict(best_model)
        print(
            f"Loaded best model from Epoch {best_epoch} "
            f"(Validation MSE = {best_val_mse:.6f})"
        )

    print(f"Company {company_idx} training terminated at epoch: {terminated_epoch}")

    # ------------------------------------------------------------------
    # Test
    # ------------------------------------------------------------------

    test_loss, test_metrics, preds, targ = eval_epoch(
        model,
        test_loader,
        device,
        loss_fn,
        mode="regression",
        company_idx=company_idx
    )

    print(
        f"Test Loss: {test_loss:.6f} | "
        f"Test Metrics: {test_metrics}"
    )

    return (
        model,
        dataset,
        price_df,
        news_labeled,
        sentiment_df,
        preds,
        targ,
        terminated_epoch,
    )

### HSBC

In [12]:
if __name__ == "__main__":

    # Quick run (may download models & price data)
    model, dataset, price_df, news_labeled, sentiment_df, preds, targ,terminated_epoch = full_news_demo(
        tickers=unique_tickers,
        start_date=apple_data["date"].min(),
        end_date=apple_data["date"].max(),
        n_news=len(apple_data),
        seq_len=5,
        co_window=5,
        batch_size=8,
        epochs=100,          # Set a large maximum
        device_str="cpu",
    )

/tmp/ipykernel_2331/3097096952.py:97: FutureWarning: YF.download() has changed argument auto_adjust default to True
  price_df = yf.download(tickers, start=start_date, end=end_date, progress=False)["Close"]
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Classifying synthetic news with FinBERT...
Sample labeled news:
         date                                              title  \
0  2024-05-08              HSBC Holdings PLC's Dividend Analysis   
1  2024-05-06  Top Three UK Dividend Stocks To Watch In May 2024   
2  2024-05-04  Ping An votes against reappointment of HSBC CE...   
3  2024-05-04  HSBC Holdings First Quarter 2024 Earnings: Bea...   
4  2024-05-03  3 Foreign Bank Stocks Worth a Look in a Prospe...   

                                            mentions     label     score  
0  ['0005.HK', 'H1SB34.SA', 'HBC1.BE', 'HBC1.DU',...   neutral  0.935243  
1  ['0005.HK', 'BYG.LSE', 'DCC.LSE', 'GFTU.LSE', ...   neutral  0.934711  
2  ['0005.HK', '2318.HK', '601318.SHG', 'H1SB34.S...   neutral  0.818103  
3  ['0005.HK', 'H1SB34.SA', 'HBC1.BE', 'HBC1.DU',...  positive  0.926698  
4  ['0005.HK', 'GSPC.INDX', 'H1SB34.SA', 'HBC1.BE...   neutral  0.701650  
Aggregating sentiment per company-day...
Sentiment df head:
            HSBC 

/usr/local/lib/python3.12/dist-packages/pandas/core/internals/blocks.py:347: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:194: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:205: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)


in_feats=3, dataset length=1761
Epoch 1/100 | Train Loss: 1.595524 | Val Loss: 4.307421 | Val MSE (Target Company): 4.307421
Validation improved. Best validation MSE = 4.307421
Epoch 2/100 | Train Loss: 1.454185 | Val Loss: 3.034893 | Val MSE (Target Company): 3.034893
Validation improved. Best validation MSE = 3.034893
Epoch 3/100 | Train Loss: 1.400538 | Val Loss: 4.187761 | Val MSE (Target Company): 4.187761
No improvement (1/5)
Epoch 4/100 | Train Loss: 1.407875 | Val Loss: 3.078824 | Val MSE (Target Company): 3.078825
No improvement (2/5)
Epoch 5/100 | Train Loss: 1.400502 | Val Loss: 3.429483 | Val MSE (Target Company): 3.429483
No improvement (3/5)
Epoch 6/100 | Train Loss: 1.400865 | Val Loss: 2.556735 | Val MSE (Target Company): 2.556735
Validation improved. Best validation MSE = 2.556735
Epoch 7/100 | Train Loss: 1.376706 | Val Loss: 2.997464 | Val MSE (Target Company): 2.997464
No improvement (1/5)
Epoch 8/100 | Train Loss: 1.370189 | Val Loss: 2.207729 | Val MSE (Target Com